# W3-A3 — what the split actually hands over

**CPU only, a few minutes, no GPU.** The measurement that turns the leakage
claim from "there is a leak" into "here is the leak, and here is why its size
depends on the corpus".

## Why it now runs on two corpora

W3-A4 ran the same recipe on DeepShip (4 classes) and OceanShip (12) and got
**0.9219** and **0.3649** on their clip-random splits. Same code, same model,
same hyper-parameters, two corpora that are both ONC Strait of Georgia
hydrophone data. The difference cannot be the split *name*.

`src/w1_05_recording_structure.py` (pure metadata, no audio) found the reason:

| | DeepShip | OceanShip |
|---|---|---|
| recordings | 609 | 10,168 |
| clips/recording, median | **51** | **3** |
| clip index is exactly 0..N-1 | **609/609** | no such index |
| (N x 5 s) / recording length | **0.994** | n/a |
| test clips with an adjacent clip in train (clip_random) | **89.6%** | 5.68% (measured, below) |

DeepShip tiles each long recording into back-to-back 5 s windows, so clip *i* and
clip *i+1* are the same audio event. OceanShip cuts short annotated segments out
of a continuous stream at arbitrary times, so two clips of one recording are
unrelated. A random split is therefore catastrophic on one corpus and nearly
inert on the other — and no dataset paper says which kind it published.

## What this notebook measures, per corpus

| cell | question |
|---|---|
| 3 | the metadata ceiling on the very splits the models were scored on |
| 4 | **1-NN**: nearest training clip — right label? same recording? same vessel? |
| 5 | among same-recording neighbours, **how far away in the stream** are they |
| 6 | are two clips of one recording closer together than two of different ones |
| 7 | duration alone, nearest class mean, linear probe |
| 8 | the money table |

Cell 8 prints the table this project now hangs on.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. locate the feature dataset
#
# Kaggle mounts a dataset by slug, in one of two layouts, and it expands
# archives on ingest. So rather than hard-coding a path we look for the one
# thing that must exist: <root>/oceanship/index.csv. If detection ever fails,
# copy the real path out of the listing below into MANUAL and re-run.
# --------------------------------------------------------------------------- #
import csv, json, os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUNS = WORK / "w3-runs"
SCRIPTS = WORK / "scripts"
MANUAL = {}          # e.g. MANUAL["features"] = "/kaggle/input/under-water-features/uwatr-features"

T0 = time.time()
def tick(msg):
    print(f"  [{time.time() - T0:7.1f}s] {msg}", flush=True)


def candidates():
    out = []
    for depth in range(1, 5):
        for p in sorted(INPUT.glob("/".join(["*"] * depth))):
            if p.is_dir() and (p / "oceanship" / "index.csv").exists():
                out.append(p)
    return sorted(set(out), key=lambda p: len(p.parts))


MANUAL_FEATURES = MANUAL.get("features")
FEAT = Path(MANUAL_FEATURES) if MANUAL_FEATURES else None
if FEAT is None:
    cands = candidates()
    FEAT = cands[0] if cands else None

print("=" * 88)
print("inputs")
print("=" * 88)
for depth in range(1, 3):
    for p in sorted(INPUT.glob("/".join(["*"] * depth))):
        if p.is_dir():
            kids = sorted(x.name for x in p.iterdir())[:6]
            print(f"  {p}")
            print(f"      {kids}")

if FEAT is None:
    print()
    print("  no feature tree found. A mounted dataset must contain")
    print("  <root>/oceanship/index.csv, <root>/deepship/index.csv, <root>/shipsear/index.csv,")
    print("  /splits. Copy the correct path into MANUAL at the top of this cell and re-run.")
    raise SystemExit("under_water_features not found")

print()
print(f"  features root : {FEAT}")
need = [FEAT / c / "index.csv" for c in ("oceanship", "deepship", "shipsear")]
need += [FEAT / c / f"{r}.npy" for c in ("oceanship", "deepship", "shipsear")
         for r in ("logmel", "spec")]
missing = [str(p.relative_to(FEAT)) for p in need if not p.exists()]
if missing:
    print(f"  MISSING: {missing}")
    raise SystemExit("feature dataset incomplete")

for c in ("oceanship", "deepship", "shipsear"):
    n = sum(1 for _ in open(FEAT / c / "index.csv", encoding="utf-8")) - 1
    gb = sum((FEAT / c / f"{r}.npy").stat().st_size for r in ("logmel", "spec")) / 1e9
    print(f"  {c:<10} {n:>7,} clips   {gb:>6.2f} GB of arrays")
print(f"  splits     : {sorted(p.name for p in (FEAT / 'splits').glob('*'))[:8]}")

mf = FEAT / "manifest.json"
if mf.exists():
    m = json.loads(mf.read_text(encoding="utf-8"))
    print(f"  manifest   : { {k: v.get('rows') for k, v in m.items() if isinstance(v, dict)} }")

SCRIPTS.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only"
    print(f"  torch      : {torch.__version__}  cuda={torch.cuda.is_available()}  {gpu}")
except ImportError:
    # W3-A3 is CPU-only and must still run where torch is absent
    print("  torch      : not installed (fine for the CPU-only notebooks)")
print(f"  free space : {shutil.disk_usage(WORK).free / 1e9:.1f} GB")
tick("setup done")


## 1. descriptors, both corpora

Mean-pool each `(128, 313)` log-mel map to `16 x 16 = 256` numbers: coarse
enough that an all-pairs search is one matrix product, fine enough to keep the
time-frequency shape a recording signature would live in.

In [ ]:
# --------------------------------------------------------------------------- #
# 2. compact descriptors, one per corpus
#
# A 128x313 map is 40,040 numbers per clip; all-pairs distances need something
# smaller.  Mean-pooling to 16x16 keeps the coarse time-frequency shape - which
# is where a recording signature would live - and turns the nearest-neighbour
# search into a single matrix product.
# --------------------------------------------------------------------------- #
import collections
import numpy as np
import pandas as pd

P_CORPUS = {
    # corpus    : label col,  vessel col,  recording col, split join keys, stream?
    #
    # `stream` says whether the index carries a POSITION IN THE AUDIO STREAM.
    # The index column clip_idx is not it: the builder writes it as a row
    # counter inside a write batch, so it is a position only by accident for
    # corpora that are processed recording by recording.  DeepShip qualifies
    # (checked below); OceanShip does not, and pretending otherwise would
    # invent an adjacency that the corpus does not have.
    "deepship":  dict(label="class_dir", vessel="ship_name", rec="rec_key",
                      keys=("rec_key", "clip_idx"), stream=True),
    "oceanship": dict(label="label", vessel="mmsi", rec="rec_key",
                      keys=("clip_id",), stream=False),
}

IDX, DES = {}, {}
for corpus, spec in P_CORPUS.items():
    idx = pd.read_csv(FEAT / corpus / "index.csv")
    IDX[corpus] = idx
    arr = np.load(FEAT / corpus / "logmel.npy", mmap_mode="r")
    n_mels, n_fr = arr.shape[1], arr.shape[2]
    pad = int(np.ceil(n_fr / 16.0) * 16)
    out = np.zeros((len(idx), 256), dtype=np.float32)
    CH = 2048
    for s in range(0, len(idx), CH):
        e = min(s + CH, len(idx))
        x = np.asarray(arr[s:e], dtype=np.float32)
        x = x.reshape(len(x), 16, n_mels // 16, n_fr).mean(2)
        if pad > n_fr:
            x = np.pad(x, ((0, 0), (0, 0), (0, pad - n_fr)), mode="edge")
        out[s:e] = x.reshape(len(x), 16, 16, pad // 16).mean(3).reshape(len(x), -1)
    DES[corpus] = out
    print(f"  {corpus:<10} {tuple(arr.shape)} {arr.dtype} -> {out.shape}   "
          f"{out.min():.1f}..{out.max():.1f}")
    np.save(WORK / f"leakprobe_{corpus}16.npy", out)

# a clip's join key, as a plain string, for both index and split table
def keys_of(corpus: str, frame) -> list:
    k = P_CORPUS[corpus]["keys"]
    if len(k) == 1:
        return frame[k[0]].astype(str).tolist()
    return [f"{a}|{b}" for a, b in zip(*(frame[c].astype(str) for c in k))]


MET = {}
for corpus in P_CORPUS:
    spec = P_CORPUS[corpus]
    ks = keys_of(corpus, IDX[corpus])
    MET[corpus] = {
        "label": dict(zip(ks, IDX[corpus][spec["label"]].astype(str))),
        "vessel": dict(zip(ks, IDX[corpus][spec["vessel"]].astype(str))),
        "rec": dict(zip(ks, IDX[corpus][spec["rec"]].astype(str))),
        "dur": dict(zip(ks, IDX[corpus]["src_dur_s"].astype(float))),
        "row": {k: i for i, k in enumerate(ks)},
    }
    for extra in ("clip_idx",):
        if extra in IDX[corpus]:
            MET[corpus][extra] = dict(
                zip(ks, IDX[corpus][extra].astype(int)))
tick("descriptors built")


## 2. the ceiling on each split

The ceiling is what the *split unit* is worth — predicting a test clip's
majority **training** label for its own recording or vessel, with no audio at
all. It is the target a leak-exploiting model would chase, and it has to be read
per corpus, because DeepShip's vessel ceiling is not OceanShip's.

In [ ]:
# --------------------------------------------------------------------------- #
# 3. the splits, and the metadata ceiling on each of them
#
# The ceiling is what the split unit is worth: predict a test clip's majority
# TRAIN label for its own recording / vessel, with no audio at all.  Reading it
# next to the achieved accuracy is the whole argument, so it is computed for
# every protocol of both corpora.
# --------------------------------------------------------------------------- #
def split_table(corpus: str):
    for name in (f"{corpus}_splits.csv.gz", f"{corpus}_splits.csv"):
        p = FEAT / "splits" / name
        if p.exists():
            return pd.read_csv(p)
    raise SystemExit(f"no {corpus} splits in the feature dataset")


SP = {}
for corpus in P_CORPUS:
    sp = split_table(corpus)
    sp = sp.assign(_k=keys_of(corpus, sp))
    SP[corpus] = sp
    print(f"  {corpus:<10} {len(sp):,} rows  protocols={sorted(sp.protocol.unique())}"
          f"  seeds={sorted(sp.seed.unique())}")


def sides(corpus: str, proto: str, seed: int):
    sp = SP[corpus]
    sel = sp[(sp.protocol == proto) & (sp.seed == int(seed))]
    have = MET[corpus]["row"]
    tr = [k for k in sel.loc[sel.split == "train", "_k"] if k in have]
    te = [k for k in sel.loc[sel.split == "test", "_k"] if k in have]
    return tr, te


def ceiling(corpus: str, tr: list, te: list, field: str):
    lab, who = MET[corpus]["label"], MET[corpus][field]
    maj = {}
    for k in tr:
        maj.setdefault(who[k], collections.Counter())[lab[k]] += 1
    maj = {k: v.most_common(1)[0][0] for k, v in maj.items()}
    cov = sum(1 for k in te if who[k] in maj)
    hit = sum(1 for k in te if maj.get(who[k]) == lab[k])
    return cov / max(len(te), 1), hit / max(len(te), 1)


PROTOCOLS = {c: [p for p in ("clip_random", "rec_random", "temporal",
                            "vessel_disjoint") if p in set(SP[c].protocol)]
             for c in P_CORPUS}
print()
print("=" * 88)
print("the metadata ceiling, on the very splits the models are scored on")
print("=" * 88)
CEIL = {}
for corpus in P_CORPUS:
    print(f"  {corpus}")
    for proto in PROTOCOLS[corpus]:
        tr, te = sides(corpus, proto, 0)
        if not te:
            continue
        cv, av = ceiling(corpus, tr, te, "vessel")
        cr, ar = ceiling(corpus, tr, te, "rec")
        maj = collections.Counter(MET[corpus]["label"][k] for k in tr).most_common(1)[0]
        CEIL[(corpus, proto)] = (av, ar)
        print(f"      {proto:<17} train {len(tr):>6,}  test {len(te):>6,}   "
              f"vessel {100*av:6.2f}% (cov {100*cv:5.1f}%)   "
              f"recording {100*ar:6.2f}%   best constant {100*maj[1]/len(tr):5.2f}%")
tick("ceilings computed")


## 3. is the identity in the features?

The decisive measurement. Each rate is printed next to its own null — the chance
that a training clip picked at random would show the same property. OceanShip has
10,168 recordings, so a 1% rate is already enormous; DeepShip has 609.

In [ ]:
# --------------------------------------------------------------------------- #
# 4. is the identity IN the features?  the same measurement, both corpora
#
# For every test clip, find its single nearest training clip in the 256-d
# descriptor and ask three questions: is the label right (is the leak usable),
# and does the neighbour share the recording / the vessel (is identity present)?
#
# Every rate is printed against its own null - the probability that the same
# event happens for a training clip picked at random.  OceanShip has 10,168
# recordings, so 1% is enormous; DeepShip has 609, so it is not.
# --------------------------------------------------------------------------- #
def distance_matrix(corpus: str, tr: list, te: list):
    rtr = np.array([MET[corpus]["row"][k] for k in tr])
    rte = np.array([MET[corpus]["row"][k] for k in te])
    T = DES[corpus][rtr]
    X = DES[corpus][rte]
    mu, sd = T.mean(0), T.std(0) + 1e-6
    T = np.ascontiguousarray((T - mu) / sd, dtype=np.float32)
    X = np.ascontiguousarray((X - mu) / sd, dtype=np.float32)
    tn = (T ** 2).sum(1)[None, :]
    nn = np.empty(len(X), dtype=np.int32)
    nnd = np.empty(len(X), dtype=np.float32)
    B = 512
    for s in range(0, len(X), B):
        e = min(s + B, len(X))
        d2 = (X[s:e] ** 2).sum(1)[:, None] + tn - 2.0 * (X[s:e] @ T.T)
        j = np.argmin(d2, 1)
        nn[s:e] = j
        nnd[s:e] = d2[np.arange(e - s), j]
    return T, X, nn, nnd


def same_null(corpus: str, tr: list, te: list, field: str) -> float:
    who = MET[corpus][field]
    cnt = collections.Counter(who[k] for k in tr)
    return sum(cnt[who[k]] for k in te) / len(te) / len(tr)


KNN = {}
print("=" * 88)
print("1-NN: does a test clip's nearest TRAINING clip share its identity?")
print("=" * 88)
for corpus in P_CORPUS:
    for proto in PROTOCOLS[corpus]:
        tr, te = sides(corpus, proto, 0)
        if not te:
            continue
        T, X, nn, nnd = distance_matrix(corpus, tr, te)
        lab, rec, ves = (MET[corpus]["label"], MET[corpus]["rec"],
                         MET[corpus]["vessel"])
        hit = np.mean([lab[tr[j]] == lab[c] for j, c in zip(nn, te)])
        sr = np.mean([rec[tr[j]] == rec[c] for j, c in zip(nn, te)])
        sv = np.mean([ves[tr[j]] == ves[c] for j, c in zip(nn, te)])
        nr = same_null(corpus, tr, te, "rec")
        nv = same_null(corpus, tr, te, "vessel")
        best = collections.Counter(lab[k] for k in tr).most_common(1)[0][1] / len(tr)
        rec_ = dict(proto=proto, seed=0, label_acc=float(hit), same_rec=float(sr),
                    same_ves=float(sv), null_rec=float(nr), null_ves=float(nv),
                    best_constant=float(best), n_train=len(tr), n_test=len(te))
        KNN[(corpus, proto)] = rec_
        print(f"  [{corpus} {proto} s0]  train {len(tr):,}  test {len(te):,}")
        print(f"      1-NN label accuracy        {100*hit:6.2f}%   "
              f"(best constant {100*best:5.2f}%)")
        print(f"      nearest neighbour shares the RECORDING {100*sr:6.2f}%   "
              f"(chance {100*nr:8.4f}%  ->  {sr/max(nr,1e-12):8.1f}x)")
        print(f"      nearest neighbour shares the VESSEL    {100*sv:6.2f}%   "
              f"(chance {100*nv:8.4f}%  ->  {sv/max(nv,1e-12):8.1f}x)")
tick("1-NN done")


## 4. why: where the nearest neighbour sits in the stream

If the nearest same-recording training clip is the **adjacent 5 s window**, the
network is not recognising ship types at all — it is retrieving a near-duplicate.
This cell measures that distance directly, and reports the case where it cannot
be measured at all as the finding it is.

In [ ]:
# --------------------------------------------------------------------------- #
# 5. WHY one corpus leaks and the other does not: adjacency in the raw stream
#
# DeepShip tiles each long recording into back-to-back 5 s windows, so a test
# clip's stream neighbours are near-duplicates of it.  OceanShip cuts short
# annotated segments out of a continuous hydrophone stream at arbitrary times,
# so two clips of one recording are unrelated audio.
#
# This cell checks that directly on the audio: among the test clips whose
# nearest training clip IS from their own recording, how far away in the stream
# is it?  If the answer is "one window", the model is doing retrieval.
# --------------------------------------------------------------------------- #
print("=" * 88)
print("where the nearest same-recording neighbour sits in the stream")
print("=" * 88)
for corpus in P_CORPUS:
    if not P_CORPUS[corpus]["stream"]:
        print(f"  {corpus:<10} the index carries no point in the audio stream -")
        print(f"  {'':<10} clip_idx in the feature index is a build counter, and the")
        print(f"  {'':<10} trailing integer of clip_id is not chronological either")
        print(f"  {'':<10} (median gap 0 between consecutive ids within a recording).")
        print(f"  {'':<10} This absence IS the finding: clips of one recording are")
        print(f"  {'':<10} arbitrary moments, so nothing about them is a neighbour.")
        print()
        continue
    ci = MET[corpus]["clip_idx"]
    # wiring check: for a corpus that IS tiled, clip_idx must run 0..N-1 per rec
    byrec = collections.defaultdict(list)
    for k, i in ci.items():
        byrec[MET[corpus]["rec"][k]].append(i)
    tiled = sum(1 for v in byrec.values() if sorted(v) == list(range(len(v))))
    print(f"  {corpus:<10} index clip_idx runs 0..N-1 per recording for "
          f"{tiled}/{len(byrec)} recordings")
    for proto in PROTOCOLS[corpus]:
        tr, te = sides(corpus, proto, 0)
        if not te:
            continue
        rec = MET[corpus]["rec"]
        _, _, nn, _ = distance_matrix(corpus, tr, te)
        delta = [abs(ci[c] - ci[tr[j]]) for j, c in zip(nn, te)
                 if rec[tr[j]] == rec[c]]
        if not delta:
            print(f"      {proto:<17} no same-recording nearest neighbour at all "
                  f"({len(te):,} test clips)")
            continue
        d = np.array(delta)
        print(f"      {proto:<17} {len(d):,} test clips with a same-recording "
              f"nearest neighbour")
        print(f"          |delta clip_idx|  = 1 {100*np.mean(d == 1):6.1f}%   "
              f"<= 2 {100*np.mean(d <= 2):6.1f}%   <= 5 {100*np.mean(d <= 5):6.1f}%"
              f"   median {np.median(d):.0f}")
        print(f"          a neighbour one window away is the SAME audio event")
    print()
tick("adjacency done")


## 5. the other two explanations

Separation first (are same-recording pairs actually closer?), then duration —
the clips were zero-padded to 5 s and the padding is not evenly spread across
classes, so a model can read the silence.

In [ ]:
# --------------------------------------------------------------------------- #
# 6. the two explanations that are not the model
#
# (a) separation: are two clips of one recording closer than two clips of
#     different recordings?  ratio << 1 means identity is present.
# (b) duration: the clips were zero-padded to 5 s and the padding is not evenly
#     spread across classes, so a model can read the silence.
# (c) a linear probe on the same descriptor, as the cheap supervised baseline.
# --------------------------------------------------------------------------- #
for corpus in P_CORPUS:
    for proto in PROTOCOLS[corpus]:
        tr, te = sides(corpus, proto, 0)
        if not te:
            continue
        T, X, nn, nnd = distance_matrix(corpus, tr, te)
        rec = MET[corpus]["rec"]
        same = np.array([rec[tr[j]] == rec[c] for j, c in zip(nn, te)])
        if same.any() and (~same).any():
            print(f"  [{corpus} {proto}] nearest-neighbour distance, normalised")
            print(f"      same recording     median {nnd[same].mean():.1f}")
            print(f"      different record.  median {nnd[~same].mean():.1f}")
            print(f"      ratio {nnd[same].mean()/nnd[~same].mean():.3f}   "
                  f"(marginally closer -> identity present)")
        else:
            print(f"  [{corpus} {proto}] no same-recording neighbours at all: "
                  f"the protocol removed them")
tick("separability done")


In [ ]:
# --------------------------------------------------------------------------- #
# 7. the scalar baseline, and a linear probe on the same descriptor
#
# Not every baseline needs audio.  DeepShip's src_dur_s is the SOURCE FILE's
# length, and the builder cuts one file into its 5 s windows - so every clip of a
# recording carries the same number, and one metadata column is then a recording
# fingerprint.  OceanShip's src_dur_s is the clip's OWN length, which varies
# inside a recording, so there is nothing to fingerprint.
#
# Both readings matter: the 1-NN asks whether duration points at the recording,
# the histogram asks whether it points at the class.
# --------------------------------------------------------------------------- #
DUR1NN = {}
for corpus in P_CORPUS:
    dur, lab = MET[corpus]["dur"], MET[corpus]["label"]
    for proto in PROTOCOLS[corpus]:
        tr, te = sides(corpus, proto, 0)
        if not tr or not te:
            continue
        cl = np.array([dur[k] for k in tr], dtype=np.float32)
        xl = np.array([dur[k] for k in te], dtype=np.float32)
        yl = np.array([lab[k] for k in tr])
        pred = np.empty(len(xl), dtype=object)
        for s0 in range(0, len(xl), 512):
            e = min(s0 + 512, len(xl))
            j = np.argmin(np.abs(xl[s0:e, None] - cl[None, :]), 1)
            pred[s0:e] = yl[j]
        DUR1NN[(corpus, proto)] = float((pred == np.array([lab[k] for k in te])).mean())

print("=" * 88)
print("1-NN on src_dur_s alone - one metadata column, no audio, no training")
print("=" * 88)
for corpus in P_CORPUS:
    lab = MET[corpus]["label"]
    for proto in PROTOCOLS[corpus]:
        if (corpus, proto) not in DUR1NN:
            continue
        tr, te = sides(corpus, proto, 0)
        best = collections.Counter(lab[k] for k in tr).most_common(1)[0][1] / len(tr)
        print(f"  {corpus:<10} {proto:<17} {100*DUR1NN[(corpus, proto)]:6.2f}%"
              f"   (best constant {100*best:5.2f}%)")
print()
print("  duration histogram -> majority label  (does duration carry the CLASS?)")
for corpus in P_CORPUS:
    proto = PROTOCOLS[corpus][0]
    tr, te = sides(corpus, proto, 0)
    lab, dur = MET[corpus]["label"], MET[corpus]["dur"]
    cl = np.array([dur[k] for k in tr], dtype=np.float32)
    edges = np.quantile(cl, np.linspace(0, 1, 21))
    edges[-1] += 1e-6
    maj = {}
    for b in range(20):
        sel = [tr[i] for i in np.where((cl >= edges[b]) & (cl < edges[b + 1]))[0]]
        if sel:
            maj[b] = collections.Counter(lab[c] for c in sel).most_common(1)[0][0]
    hit = sum(maj.get(int(np.clip(np.searchsorted(edges, dur[c]) - 1, 0, 19))) == lab[c]
              for c in te)
    print(f"    {corpus:<10} {proto:<17} {100*hit/len(te):6.2f}%")
print()

# --------------------------------------------------------------------------- #
# 8. the supervised floor on the same descriptor: nearest class mean, then a
#    ridge-regularised linear probe (closed form, so no GPU is involved)
# --------------------------------------------------------------------------- #
for corpus in P_CORPUS:
    proto = PROTOCOLS[corpus][0]
    tr, te = sides(corpus, proto, 0)
    lab = MET[corpus]["label"]
    rows_tr = np.array([MET[corpus]["row"][k] for k in tr])
    rows_te = np.array([MET[corpus]["row"][k] for k in te])
    # NB: not T0 - that name belongs to the timer set up in the setup cell
    Td, Xd = DES[corpus][rows_tr], DES[corpus][rows_te]
    mu, sd = Td.mean(0), Td.std(0) + 1e-6
    T = np.ascontiguousarray((Td - mu) / sd, dtype=np.float32)
    X = np.ascontiguousarray((Xd - mu) / sd, dtype=np.float32)
    classes = sorted({lab[c] for c in tr})
    ci = {k: i for i, k in enumerate(classes)}
    Y = np.array([ci[lab[c]] for c in tr])
    Xy = np.array([ci.get(lab[c], -1) for c in te])
    M = np.stack([T[Y == k].mean(0) for k in range(len(classes))])
    ncm = np.mean(np.argmin(((X[:, None, :] - M[None, :, :]) ** 2).sum(2), 1) == Xy)
    Yh = np.eye(len(classes), dtype=np.float32)[Y]
    W = np.linalg.solve(T.T @ T + 1e-2 * np.eye(T.shape[1], dtype=np.float32),
                        T.T @ Yh)
    lin = np.mean((X @ W).argmax(1) == Xy)
    print(f"  [{corpus} {proto}]  nearest class mean (256-d) {100*ncm:6.2f}%   "
          f"linear probe (ridge) {100*lin:6.2f}%   ({len(classes)} classes)")
tick("scalar + linear probe done")


In [ ]:
# --------------------------------------------------------------------------- #
# 9. the table this whole notebook exists to produce
#
# Model accuracies are entered by hand from the run logs (results.csv on Kaggle
# keeps them; this notebook is CPU-only and deliberately reads no runs).  Update
# them when a sweep is re-run - the point is the ORDER, not the third decimal.
# --------------------------------------------------------------------------- #
MODEL = {
    # (corpus, protocol): test accuracy of logmel/cnn_small at the same recipe
    ("deepship", "clip_random"): 0.9219,
    ("deepship", "rec_random"): 0.6842,
    ("deepship", "temporal"): 0.6182,
    ("oceanship", "clip_random"): 0.3649,
    ("oceanship", "vessel_disjoint"): 0.2926,
    # OceanShip rec_random has not been run at this recipe: it prints as "-"
    # rather than being guessed.  W3-A fills it in for all five seeds.
}

print("=" * 88)
print("the money table: what the split hands over, and what the model converts")
print("=" * 88)
print()
print(f"  {'corpus':<11}{'protocol':<17}{'meta ceiling':>13}{'1 scalar':>10}"
      f"{'1-NN label':>12}{'1-NN same rec':>14}{'chance':>9}{'model':>8}")
print("  " + "-" * 92)
for corpus in P_CORPUS:
    for proto in PROTOCOLS[corpus]:
        if (corpus, proto) not in KNN:
            continue
        k = KNN[(corpus, proto)]
        m = MODEL.get((corpus, proto))
        s = DUR1NN.get((corpus, proto))
        s_txt = f"{100*s:>9.2f}%" if s is not None else f"{'-':>10}"
        m_txt = f"{m:>8.4f}" if m is not None else f"{'-':>8}"
        print(f"  {corpus:<11}{proto:<17}{100*CEIL[(corpus, proto)][1]:>12.2f}%"
              f"{s_txt}{100*k['label_acc']:>12.2f}%{100*k['same_rec']:>13.2f}%"
              f"{100*k['null_rec']:>8.3f}%{m_txt}")
    print()

print("  how to read the two blocks")
print()
print("    deepship, clip_random   every clip is one of ~51 back-to-back 5 s")
print("                            windows tiled out of a single file, so this")
print("                            split leaves ~90% of test clips with a")
print("                            near-duplicate in train.  ONE metadata column")
print("                            (source-file length) already gets 78% with no")
print("                            audio and no training; the full 256-d")
print("                            descriptor gets 83%; the trained model 92%.")
print("                            That ordering is the whole argument: the")
print("                            headline number is a retrieval task.")
print()
print("    deepship, rec_/temporal  the scalar collapses to the best-constant")
print("                            level, and so does the same-recording rate -")
print("                            which is what a split is supposed to do.")
print()
print("    oceanship, *            the recording ceiling is 90.8% on clip_random")
print("                            and the vessel ceiling 99.5%, yet the 1-NN")
print("                            same-recording rate is a few percent, and the")
print("                            scalar is no better than a constant.  Clips of")
print("                            one recording are arbitrary moments of a")
print("                            hydrophone stream, so the split hands over an")
print("                            identity the ACOUSTICS never expose.")
print()
print("  which is why the same code, model and recipe produce a 30.4 pp gap on one")
print("  corpus and a 7.2 pp gap on the other.  'We used a random split' is not a")
print("  specification: the size of the leak is set by how the corpus cuts its")
print("  recordings into clips, and no dataset paper states it.")
tick("verdict done")


## 6. keep the descriptors

`/kaggle/working/leakprobe_<corpus>16.npy` are 256-d for every clip of both
corpora — a few tens of MB. Create a dataset from the output if you want to
re-run this analysis without rebuilding them.

In [ ]:
# --------------------------------------------------------------------------- #
# keep the numbers and the descriptors
#
# W3-A3 writes no runs, so there is no results.csv to carry forward.  What is
# worth keeping is the plain record of the measurements - a later session can
# re-run the whole analysis from these numbers - and the descriptors themselves,
# so the nearest-neighbour work does not have to read 14 GB of float16 again.
# --------------------------------------------------------------------------- #
import json

PROBE = WORK / "w3a3"
PROBE.mkdir(parents=True, exist_ok=True)

record = {
    "kind": "w3a3-leak-probe",
    "notebook": "W3-A3",
    "descriptor": "mean-pooled log-mel 16x16 (256 dims), per corpus",
    "metadata_ceiling": {
        f"{c}|{p}": {"vessel": CEIL[(c, p)][0], "recording": CEIL[(c, p)][1]}
        for (c, p) in CEIL},
    "knn": {f"{c}|{p}": v for (c, p), v in KNN.items()},
    "note": "CPU only, no runs written; descriptors saved as "
            "leakprobe_<corpus>16.npy so the analysis can be repeated without "
            "re-reading the feature arrays",
}
(PROBE / "w3a3_probe.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2, default=float)[:1200])
print()
print("  -> Output tab -> Create Dataset if you want the descriptors kept")
